# CALCE study XLSX → reconstructed targets → archived results (Step 11A / 11O)

**Scientific aim.** Verify all six constant-protocol CALCE cells (CS2_33–CS2_38) and two variable-protocol cells (CS2_9, CS2_3) directly against the original 231 Excel measurement workbooks inside `collab.zip` in Google Drive. This is an **audit**, not a new model-fit experiment.

- Constant: replicate the published Step 11A right-sample negative-current time integral and first-valid-discharge Ah reference.
- Variable: replicate the published Step 11A current-based 0.22 ± 0.04 A reference-block selection and trapezoidal integral.
- Preserve acquisition-date file order, worksheet sequence, cumulative `Cycle_Index` offsets, QC, raw candidate cycles and all target comparisons.
- Check the 5,663 QC-valid CALCE constant targets against saved Step 11O labels.
- Record SHA-256 hashes of all 231 source XLSX files (without redistributing the workbooks), and save detailed reports to Google Drive.

The complete Step 11A original functions and frozen audit CSV files are embedded below; no GitHub login or extra uploads are needed. The original code and reference exports must not be edited to make a comparison pass. This audit **does not** establish historical Excel-generation provenance or complete model retraining.

In [ ]:
%pip -q install openpyxl


In [ ]:
from google.colab import drive
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime, timezone
from collections import Counter
import base64, gc, hashlib, io, json, os, re, shutil, sys, tempfile, traceback, zipfile
import numpy as np
import pandas as pd

drive.mount('/content/drive')
RAW_ARCHIVE = Path('/content/drive/MyDrive/Battery_SOH_Project/raw_data/collab.zip')
OUT_DIR = Path('/content/drive/MyDrive/Battery_SOH_Project/reviewer_response/calce_source_verification')
LOCAL_ROOT = Path('/content/calce_source_verification')
CACHE_DIR = OUT_DIR/'checkpoints_step11a_exact'
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_ARCHIVE.is_file(), f'Could not find {RAW_ARCHIVE}'
print('Archive:',RAW_ARCHIVE, 'size:',RAW_ARCHIVE.stat().st_size)

# Self-contained frozen evidence from Step 11A and Step 11O; not derived from the live input.
FROZEN_BUNDLE_BASE64 = """..."""
bundle_bytes=base64.b64decode(FROZEN_BUNDLE_BASE64)
FROZEN_ZIP=LOCAL_ROOT/'frozen_step11a_step11o_evidence.zip'
FROZEN_ZIP.write_bytes(bundle_bytes)
with ZipFile(FROZEN_ZIP) as z:
    for name in z.namelist():
        (LOCAL_ROOT/name).write_bytes(z.read(name))
print('Frozen audit bundle SHA256:',hashlib.sha256(bundle_bytes).hexdigest())


In [ ]:
ORIGINAL_STEP11A_FUNCTIONS = '''# Selected functions are the original Step 11A CALCE target-reconstruction functions,
# reproduced verbatim from Reviewer_Step11A_Target_Audit_Colab.ipynb.
SOH_MIN, SOH_MAX = 70.0, 100.0
REF_I = 0.22
REF_I_TOL = 0.04
DISCHARGE_I_THRESHOLD = 0.02
MIN_BLOCK_SAMPLES = 5
CYCLE, BATTERY, SOURCE, TARGET = 'Cycle_Index', 'battery_id', 'source_dataset', 'Calculated_SOH(%)'

def norm(s):
    return ''.join(ch.lower() for ch in str(s) if ch.isalnum())

def find_col(df, names, required=True):
    lookup = {norm(c): c for c in df.columns}
    for name in names:
        if norm(name) in lookup:
            return lookup[norm(name)]
    if required:
        raise KeyError(f'None of these columns found: {names}')
    return None

def natural_key(v):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)', str(v))]

def parse_calce_date(path):
    parts = Path(path).stem.split('_')
    for start in range(len(parts)-3, -1, -1):
        m,d,y = parts[start:start+3]
        if not (m.isdigit() and d.isdigit() and y.isdigit()) or len(y) not in (2,4):
            continue
        yy = int(y) + (2000 if len(y)==2 else 0)
        try:
            return pd.Timestamp(year=yy, month=int(m), day=int(d))
        except ValueError:
            pass
    return None

def calce_files(folder):
    folder = Path(folder)
    fs = [p for p in folder.rglob('*.xlsx') if not p.name.startswith('~$')]
    if not fs:
        raise FileNotFoundError(f'No xlsx files in {folder}')
    if len(fs)>1:
        bad = [p for p in fs if parse_calce_date(p) is None]
        if bad:
            raise ValueError(f'Cannot parse CALCE workbook date(s): {[p.name for p in bad]}')
    fs.sort(key=lambda p: (parse_calce_date(p) is None, parse_calce_date(p) or pd.Timestamp.max, str(p).lower()))
    return fs

def read_calce(folder, sheets):
    frames, last = [], 0.0
    for path in calce_files(folder):
        with pd.ExcelFile(path) as xf:
            for sheet in sheets:
                if sheet not in xf.sheet_names:
                    continue
                d = pd.read_excel(xf, sheet_name=sheet)
                if CYCLE not in d.columns:
                    raise KeyError(f'{path.name}/{sheet}: missing {CYCLE}')
                d[CYCLE] = pd.to_numeric(d[CYCLE], errors='coerce')
                d = d[d[CYCLE].notna()].copy()
                if d.empty:
                    continue
                d[CYCLE] = d[CYCLE] + last
                last = float(d[CYCLE].max())
                frames.append(d)
    if not frames:
        raise RuntimeError(f'No usable CALCE sheets in {folder}')
    return pd.concat(frames, ignore_index=True)

def time_qc(values):
    t = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(float)
    t = np.sort(t[np.isfinite(t)])
    dup = int(pd.Series(t).duplicated().sum()) if len(t) else 0
    if len(t)<2:
        return dict(Finite_Time_Samples=len(t), Duplicate_Timestamps=dup,
                    Positive_DT_Intervals=0, Nonpositive_DT_Intervals=0,
                    Median_Positive_DT_s=np.nan, P95_Positive_DT_s=np.nan, Max_Positive_DT_s=np.nan)
    dt = np.diff(t); dt = dt[np.isfinite(dt)]; pos = dt[dt>0]
    return dict(Finite_Time_Samples=len(t), Duplicate_Timestamps=dup,
                Positive_DT_Intervals=int((dt>0).sum()), Nonpositive_DT_Intervals=int((dt<=0).sum()),
                Median_Positive_DT_s=float(np.median(pos)) if len(pos) else np.nan,
                P95_Positive_DT_s=float(np.quantile(pos,.95)) if len(pos) else np.nan,
                Max_Positive_DT_s=float(np.max(pos)) if len(pos) else np.nan)

def constant_targets(raw,b,src):
    d=raw[[CYCLE,'Test_Time(s)','Current(A)']].copy()
    for c in [CYCLE,'Test_Time(s)','Current(A)']: d[c]=pd.to_numeric(d[c],errors='coerce')
    rows=[]
    for cyc,g0 in d.groupby(CYCLE,sort=True,dropna=True):
        ns=len(g0); mt=g0['Test_Time(s)'].isna().sum(); mi=g0['Current(A)'].isna().sum()
        g=g0.dropna(subset=['Test_Time(s)','Current(A)']).sort_values('Test_Time(s)')
        t=g['Test_Time(s)'].to_numpy(float); i=g['Current(A)'].to_numpy(float)
        finite=np.isfinite(t)&np.isfinite(i); t=t[finite]; i=i[finite]; diag=time_qc(t)
        if len(t)<2:
            rows.append({BATTERY:b,SOURCE:src,'Protocol':'CONSTANT',CYCLE:float(cyc),
                         'N_Source_Rows':ns,'Missing_Time_Rows':mt,'Missing_Current_Rows':mi,
                         'N_Finite_Time_Current':len(t),'Duration_s':np.nan,
                         'Measured_Discharge_Capacity_Ah':np.nan,'Raw_QC_Pass':False,
                         'Raw_QC_Reason':'fewer_than_2_finite_samples',**diag}); continue
        order=np.argsort(t); t=t[order]; i=i[order]
        dt=np.diff(t,prepend=t[0]); dt=np.where(np.isfinite(dt)&(dt>0),dt,0.0)
        q=float(np.sum(np.maximum(-i,0.0)*dt)/3600.0); dur=float(t.max()-t.min())
        qc=bool(np.isfinite(q) and q>0 and np.isfinite(dur) and dur>0)
        rows.append({BATTERY:b,SOURCE:src,'Protocol':'CONSTANT',CYCLE:float(cyc),
                     'N_Source_Rows':ns,'Missing_Time_Rows':mt,'Missing_Current_Rows':mi,
                     'N_Finite_Time_Current':len(t),'Duration_s':dur,
                     'Measured_Discharge_Capacity_Ah':q,'Raw_QC_Pass':qc,
                     'Raw_QC_Reason':'' if qc else 'nonpositive_or_nonfinite_capacity_or_duration',**diag})
    a=pd.DataFrame(rows).sort_values(CYCLE).reset_index(drop=True)
    valid=a['Raw_QC_Pass']; qref=float(a.loc[valid,'Measured_Discharge_Capacity_Ah'].iloc[0])
    a['Reference_Capacity_Ah']=qref
    a[TARGET]=a['Measured_Discharge_Capacity_Ah']/qref*100.0
    a['Below_70']=valid&(a[TARGET]<SOH_MIN)
    a['In_Primary_70_100_Scope']=valid&a[TARGET].between(SOH_MIN,SOH_MAX,inclusive='both')
    a['Above_100']=valid&(a[TARGET]>SOH_MAX)
    a['Target_Construction']='RAW_DISCHARGE_CURRENT_TIME_INTEGRATION'
    a['Integration_Method']='Step5 validated rectangular/right-sample sum: sum(max(-I,0)*positive_dt)/3600'
    a['Current_Sign_Convention']='NASA discharge-only current -> -abs(source current)' if src=='NASA' else 'CALCE negative current = discharge'
    a['Gap_Treatment']='All finite positive dt retained; no gap-length exclusion; nonpositive dt contributes zero'
    return a

def trapz_block(block):
    b=block[['Test_Time(s)','Current(A)']].copy()
    b['Test_Time(s)']=pd.to_numeric(b['Test_Time(s)'],errors='coerce'); b['Current(A)']=pd.to_numeric(b['Current(A)'],errors='coerce')
    b=b.dropna().sort_values('Test_Time(s)')
    if len(b)<2: return np.nan,np.nan
    t=b['Test_Time(s)'].to_numpy(float); i=np.abs(b['Current(A)'].to_numpy(float))
    dt=np.diff(t); imid=.5*(i[:-1]+i[1:]); ok=np.isfinite(dt)&np.isfinite(imid)&(dt>0)
    if not ok.any(): return np.nan,np.nan
    return float(np.sum(imid[ok]*dt[ok])/3600), float(np.sum(dt[ok]))

def calce_variable_targets(raw,b):
    d=raw[[CYCLE,'Test_Time(s)','Current(A)']].copy(); d['_row']=np.arange(len(d))
    for c in [CYCLE,'Test_Time(s)','Current(A)']: d[c]=pd.to_numeric(d[c],errors='coerce')
    rows=[]
    for cyc,g0 in d.groupby(CYCLE,sort=True,dropna=True):
        ns=len(g0); mt=g0['Test_Time(s)'].isna().sum(); mi=g0['Current(A)'].isna().sum()
        g=g0.dropna(subset=['Test_Time(s)','Current(A)']).sort_values(['Test_Time(s)','_row']).copy()
        g['_dis']=g['Current(A)'] < -abs(DISCHARGE_I_THRESHOLD)
        g['_start']=g['_dis'] & ~g['_dis'].shift(fill_value=False); g['_block']=g['_start'].cumsum()
        candidates=[]
        for bid,blk in g.loc[g['_dis']].groupby('_block',sort=True):
            if len(blk)<MIN_BLOCK_SAMPLES: continue
            med=float(np.nanmedian(np.abs(blk['Current(A)'].to_numpy(float))))
            q,dur=trapz_block(blk); diag=time_qc(blk['Test_Time(s)'])
            candidates.append(dict(block=int(bid),samples=len(blk),median=med,error=abs(med-REF_I) if np.isfinite(med) else np.nan,capacity=q,duration=dur,**diag))
        valid=[x for x in candidates if np.isfinite(x['capacity']) and x['capacity']>0 and np.isfinite(x['median'])]
        chosen=min(valid,key=lambda x:x['error']) if valid else None
        if chosen is None:
            status='no_valid_discharge_block'; chosen=dict(block=np.nan,samples=np.nan,median=np.nan,error=np.nan,capacity=np.nan,duration=np.nan,**time_qc([]))
        elif chosen['error']>REF_I_TOL:
            status='closest_block_outside_current_tolerance'; chosen=dict(chosen); chosen['capacity']=np.nan
        else: status='valid_0.22A_reference'
        rows.append({BATTERY:b,SOURCE:'CALCE','Protocol':'VARIABLE',CYCLE:float(cyc),
                     'N_Source_Rows':ns,'Missing_Time_Rows':mt,'Missing_Current_Rows':mi,
                     'Candidate_Discharge_Blocks':len(candidates),'Selected_Block':chosen['block'],
                     'Selected_Samples':chosen['samples'],'Reference_Target_Current_A':REF_I,
                     'Reference_Median_Current_A':chosen['median'],'Reference_Current_Error_A':chosen['error'],
                     'Selected_Duration_s':chosen['duration'],'Reference_Capacity_Ah':chosen['capacity'],
                     'Selection_Status':status,'Raw_QC_Pass':status=='valid_0.22A_reference',
                     **{k:chosen[k] for k in time_qc([]).keys()}})
    a=pd.DataFrame(rows).sort_values(CYCLE).reset_index(drop=True)
    valid=a['Raw_QC_Pass']&np.isfinite(a['Reference_Capacity_Ah']); first=a.index[valid][0]; qref=float(a.loc[first,'Reference_Capacity_Ah'])
    a['Baseline_Reference_Capacity_Ah']=qref; a[TARGET]=np.nan
    a.loc[valid,TARGET]=a.loc[valid,'Reference_Capacity_Ah']/qref*100.0; a.loc[first,TARGET]=100.0
    a['Below_70']=valid&(a[TARGET]<SOH_MIN); a['In_Primary_70_100_Scope']=valid&a[TARGET].between(SOH_MIN,SOH_MAX,inclusive='both'); a['Above_100']=valid&(a[TARGET]>SOH_MAX)
    a['Eligible_Independent_Observation']=valid; a['Primary_Independent_Observation']=valid&a['In_Primary_70_100_Scope']
    a['Target_Construction']='RAW_0p22A_REFERENCE_CURRENT_INTEGRATION_NO_TARGET_CLEANING'
    a['Integration_Method']='Trapezoidal abs(current) integration over finite positive dt in selected 0.22 A block'
    a['Current_Sign_Convention']='CALCE negative current identifies discharge; capacity uses abs(current)'
    a['Gap_Treatment']='All finite positive dt retained; no gap-length exclusion; nonpositive dt excluded'
    return a
'''
exec(ORIGINAL_STEP11A_FUNCTIONS)
print('Audited Step11A functions SHA256:',hashlib.sha256(ORIGINAL_STEP11A_FUNCTIONS.encode()).hexdigest())


In [ ]:
def comparator(actual,expected,battery):
    # Compare all common audited fields, not only the target or reference capacity.
    keys=['Cycle_Index']
    assert not actual.duplicated(keys).any(),f'{battery}: duplicate reconstructed keys'
    assert not expected.duplicated(keys).any(),f'{battery}: duplicate archived keys'
    actual=actual.copy();expected=expected.copy()
    actual['Cycle_Index']=pd.to_numeric(actual['Cycle_Index'],errors='raise')
    expected['Cycle_Index']=pd.to_numeric(expected['Cycle_Index'],errors='raise')
    joined=actual.merge(expected,on=keys,how='outer',indicator=True,validate='one_to_one',suffixes=('_run','_arch'))
    differences=[]
    for _,r in joined.loc[~joined['_merge'].eq('both')].iterrows():
        differences.append({'battery_id':battery,'Cycle_Index':r['Cycle_Index'],'field':'_row_key','regenerated':r['_merge'],'archived':'missing','problem':'missing_row'})
    number_fields={'N_Source_Rows','Missing_Time_Rows','Missing_Current_Rows','N_Finite_Time_Current','Duration_s',
      'Measured_Discharge_Capacity_Ah','Reference_Capacity_Ah','Finite_Time_Samples','Duplicate_Timestamps',
      'Positive_DT_Intervals','Nonpositive_DT_Intervals','Median_Positive_DT_s','P95_Positive_DT_s',
      'Max_Positive_DT_s','Candidate_Discharge_Blocks','Selected_Block','Selected_Samples',
      'Reference_Target_Current_A','Reference_Median_Current_A','Reference_Current_Error_A',
      'Selected_Duration_s','Baseline_Reference_Capacity_Ah','Calculated_SOH(%)'}
    bool_fields={'Raw_QC_Pass','Below_70','In_Primary_70_100_Scope','Above_100',
                 'Eligible_Independent_Observation','Primary_Independent_Observation'}
    text_fields={'Raw_QC_Reason','Selection_Status','Target_Construction','Integration_Method',
                 'Current_Sign_Convention','Gap_Treatment'}
    common_cols=(set(actual.columns)&set(expected.columns)) - set(keys)
    for col in sorted((number_fields|bool_fields|text_fields)&common_cols):
        a=joined.loc[joined['_merge'].eq('both'),col+'_run']
        b=joined.loc[joined['_merge'].eq('both'),col+'_arch']
        if col in number_fields:
            x=pd.to_numeric(a,errors='coerce').to_numpy(float)
            y=pd.to_numeric(b,errors='coerce').to_numpy(float)
            # Values are outputs from identical formulas; tolerances cover only rounding.
            tol=5e-7 if col=='Calculated_SOH(%)' else (1e-5 if 'Time' in col or 'Duration' in col else 5e-9)
            ok=np.isclose(x,y,atol=tol,rtol=1e-10,equal_nan=True)
        elif col in bool_fields:
            x=a.fillna('NA').astype(str).str.strip().str.lower().to_numpy()
            y=b.fillna('NA').astype(str).str.strip().str.lower().to_numpy()
            ok=(x==y)
        else:
            x=a.fillna('').astype(str).str.strip().to_numpy()
            y=b.fillna('').astype(str).str.strip().to_numpy()
            ok=(x==y)
        sub=joined.loc[joined['_merge'].eq('both')]
        for (_,row),f in zip(sub.loc[~ok].iterrows(),np.flatnonzero(~ok)):
            differences.append({'battery_id':battery,'Cycle_Index':row['Cycle_Index'],'field':col,
                'regenerated':str(x[f]),'archived':str(y[f]),'problem':'value_difference'})
    return {'battery_id':battery,'passed':len(differences)==0,'regenerated_rows':len(actual),
        'archived_rows':len(expected),'difference_count':len(differences),'differences':differences}


In [ ]:
# Build exactly one record per battery; reuse previously completed battery results
# only when the input ZIP member manifest and original audit match.
CALCE_CONSTANT = [
  ('CS2_33',['Channel_1-006']), ('CS2_34',['Channel_1-007']),
  ('CS2_35',['Channel_1-008']), ('CS2_36',['Channel_1-009']),
  ('CS2_37',['Channel_1-010']), ('CS2_38',['Channel_1-011']),
]
CALCE_VARIABLE = [
  ('CS2_9',['Channel_1-007','Channel_1-007_1','Channel_1-007_2']),
  ('CS2_3',['Channel_1-008']),
]
ALL_BATTERIES = CALCE_CONSTANT + CALCE_VARIABLE
AUDIT_CONST = pd.read_csv(LOCAL_ROOT/'step11a_constant_cycle_capacity_audit.csv', low_memory=False)
AUDIT_VAR = pd.read_csv(LOCAL_ROOT/'step11a_calce_variable_reference_audit.csv', low_memory=False)
AUDIT_REGISTRY = pd.read_csv(LOCAL_ROOT/'step11a_source_input_registry.csv',low_memory=False)
AUDIT_SUMMARY = pd.read_csv(LOCAL_ROOT/'step11a_battery_reference_summary.csv',low_memory=False)
STEP11O_TARGETS = pd.read_csv(LOCAL_ROOT/'step11o_constant_6299_soh_target_values.csv')

OUTPUT_RUN=LOCAL_ROOT/'outputs'; OUTPUT_RUN.mkdir(exist_ok=True)
SOURCE_META=[]
RESULTS=[]

# Match records using battery folder anywhere in the archive; detect ambiguity.
with ZipFile(RAW_ARCHIVE) as z:
    meta=z.infolist()
    all_calce=[i for i in meta if i.filename.lower().endswith('.xlsx') and not i.is_dir() and not Path(i.filename).name.startswith('~$')]
    print('CALCE Excel files discovered:',len(all_calce))
    assert len(all_calce)==231,f'Unexpected CALCE workbook count {len(all_calce)} (expected 231)'
    archive_sig=hashlib.sha256('\n'.join(sorted(f'{x.filename}|{x.file_size}|{x.CRC}' for x in all_calce)).encode()).hexdigest()
    print('Archive member signature:',archive_sig)
    for battery,sheets in ALL_BATTERIES:
        members=[]
        for info in all_calce:
            parents=Path(info.filename).parts[:-1]
            if battery.lower() in [p.lower() for p in parents]:
                members.append(info)
        registry=AUDIT_REGISTRY.loc[AUDIT_REGISTRY.Battery.eq(battery)].iloc[0]
        expected_names=[s.strip() for s in str(registry['Workbook_Order']).split('|')]
        actual_names=[Path(m.filename).name for m in members]
        assert len(members)==int(registry['File_Count']),f'{battery}: expected {registry["File_Count"]} files; found {len(members)}'
        assert len(set(actual_names))==len(actual_names),f'{battery}: duplicate Excel basenames'
        assert set(expected_names)==set(actual_names), f'{battery}: source workbook filenames disagree with archived Step 11A registry'
        assert all(parse_calce_date(Path(n)) is not None for n in actual_names),f'{battery}: non-date filename'
        key=hashlib.sha256((archive_sig+'|'+battery+'|'+hashlib.sha256(str(registry['Workbook_Order']).encode()).hexdigest()).encode()).hexdigest()
        checkpoint=CACHE_DIR/f'{battery}_{key[:20]}'
        cp_csv=checkpoint/'regenerated.csv'; cp_src=checkpoint/'source_files.csv'
        if cp_csv.exists() and cp_src.exists():
            print(f'\n===== {battery}: using previously completed source reconstruction =====')
            reproduced=pd.read_csv(cp_csv,low_memory=False)
            SOURCE_META.append(pd.read_csv(cp_src))
        else:
            print(f'\n===== {battery}: extracting {len(members)} Excel workbooks =====')
            with tempfile.TemporaryDirectory(prefix=f'calce_{battery}_',dir=LOCAL_ROOT) as tmp:
                workfolder=Path(tmp)/battery;workfolder.mkdir()
                sha_rows=[]
                for m in members:
                    destination=workfolder/Path(m.filename).name
                    h=hashlib.sha256()
                    with z.open(m) as f, destination.open('wb') as out:
                        for chunk in iter(lambda:f.read(1<<20),b''):
                            out.write(chunk);h.update(chunk)
                    sha_rows.append(dict(battery_id=battery,archive_member=m.filename, workbook=destination.name,
                                         size_bytes=m.file_size,crc32=hex(m.CRC),sha256=h.hexdigest(),
                                         acquisition_date=str(parse_calce_date(destination).date())))
                # Step11A original functions preserve sheet order and cumulative cycle offsets.
                raw=read_calce(workfolder,sheets)
                if battery in dict(CALCE_CONSTANT):
                    reproduced=constant_targets(raw,battery,'CALCE')
                else:
                    reproduced=calce_variable_targets(raw,battery)
                del raw; gc.collect()
                checkpoint.mkdir(parents=True,exist_ok=True)
                reproduced.to_csv(cp_csv,index=False)
                pd.DataFrame(sha_rows).to_csv(cp_src,index=False)
                SOURCE_META.append(pd.DataFrame(sha_rows))
        expected=(AUDIT_CONST if battery in dict(CALCE_CONSTANT) else AUDIT_VAR)
        expected=expected[expected.battery_id.eq(battery)].copy()
        # Full audit comparison before summary metrics are reported.
        cmp=comparator(reproduced,expected,battery)
        cmp_path=OUTPUT_RUN/f'{battery}_comparison.csv'
        pd.DataFrame(cmp['differences']).to_csv(cmp_path,index=False)
        reproduced.to_csv(OUTPUT_RUN/f'{battery}_regenerated_targets.csv',index=False)
        qref_field='Reference_Capacity_Ah' if battery in dict(CALCE_CONSTANT) else 'Baseline_Reference_Capacity_Ah'
        qref=float(reproduced[qref_field].dropna().iloc[0])
        summaryrow=AUDIT_SUMMARY.loc[AUDIT_SUMMARY.Battery.eq(battery)].iloc[0]
        qref_expected=float(summaryrow.Qref_Ah)
        cmp['qref_ah']=qref;cmp['qref_expected_ah']=qref_expected
        cmp['reference_capacity_match']=bool(np.isclose(qref,qref_expected,atol=5e-9,rtol=1e-10))
        cmp['pre_scope_observations']=int(reproduced['Raw_QC_Pass'].astype(str).str.lower().eq('true').sum())
        cmp['primary_70_100']=int(reproduced['In_Primary_70_100_Scope'].astype(str).str.lower().eq('true').sum())
        cmp['expected_raw_qc_pass']=int(summaryrow.Raw_QC_Pass)
        cmp['expected_primary_70_100']=int(summaryrow.Primary_70_100)
        cmp['passed']=bool(cmp['passed'] and cmp['reference_capacity_match'] and cmp['pre_scope_observations']==cmp['expected_raw_qc_pass'] and cmp['primary_70_100']==cmp['expected_primary_70_100'])
        RESULTS.append({k:v for k,v in cmp.items() if k!='differences'})
        print('Status:', 'PASS' if cmp['passed'] else 'FAIL', '| Qref Ah:',qref,'| rows:',cmp['regenerated_rows'],'| mismatches:',cmp['difference_count'])

# Step11A => Step11O target-label linkage for every QC-valid CALCE constant cycle.
all_regenerated=[]
for battery,_ in CALCE_CONSTANT:
    p=OUTPUT_RUN/f'{battery}_regenerated_targets.csv'
    a=pd.read_csv(p)
    a=a[a.Raw_QC_Pass.astype(str).str.lower().eq('true')]
    all_regenerated.append(a[['battery_id','Cycle_Index','Calculated_SOH(%)']])
recon=pd.concat(all_regenerated,ignore_index=True)
target=STEP11O_TARGETS.loc[STEP11O_TARGETS.source_dataset.eq('CALCE'),['battery_id','Cycle_Index','Calculated_SOH(%)']].copy()
link=recon.merge(target,on=['battery_id','Cycle_Index'],how='outer',validate='one_to_one',indicator=True,suffixes=('_source','_step11o'))
common=link['_merge'].eq('both')
target_dif=(pd.to_numeric(link.loc[common,'Calculated_SOH(%)_source'],errors='coerce')-
            pd.to_numeric(link.loc[common,'Calculated_SOH(%)_step11o'],errors='coerce')).abs()
link_ok=bool(len(recon)==len(target)==5663 and common.all() and (target_dif.fillna(np.inf)<=5e-7).all())
link_report={'passed':link_ok,'regenerated_calce_qc_valid':len(recon),'archived_step11o_calce_targets':len(target),
             'matching_keys':int(common.sum()),'missing_keys':int((~common).sum()),
             'max_abs_soh_difference':float(target_dif.max()) if len(target_dif) else None}
link.to_csv(OUTPUT_RUN/'CALCE_Constant_Step11O_Label_Comparison.csv',index=False)
SOURCE_META=pd.concat(SOURCE_META,ignore_index=True)
SOURCE_META.to_csv(OUTPUT_RUN/'CALCE_Source_Workbook_SHA256.csv',index=False)
summary=pd.DataFrame(RESULTS)
summary.to_csv(OUTPUT_RUN/'CALCE_Source_Verification_Summary.csv',index=False)
(OUTPUT_RUN/'CALCE_Step11O_Link_Report.json').write_text(json.dumps(link_report,indent=2))
print('\nCALCE Source comparisons:',int(summary.passed.sum()),'/',len(summary))
print('CALCE Source-to-Step11O label checks:',link_report)
print(summary[['battery_id','passed','regenerated_rows','difference_count','qref_ah','pre_scope_observations','primary_70_100']].to_string(index=False))


In [ ]:
# Save evidence even on failure. Never modify frozen Step11A/Step11O input exports.
run_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
result_zip=OUT_DIR/f'CALCE_Source_to_Archived_Chain_{run_stamp}.zip'
report={'timestamp_utc':run_stamp,'source_archive_path':str(RAW_ARCHIVE),
        'calce_workbook_count':len(SOURCE_META),'source_workbook_manifest_sha256':archive_sig,
        'code_sha256':hashlib.sha256(ORIGINAL_STEP11A_FUNCTIONS.encode()).hexdigest(),
        'battery_results':RESULTS,'step11o_label_link':link_report,
        'all_pass':bool(all(r['passed'] for r in RESULTS) and link_ok),
        'scope':'CALCE study XLSX -> Step11A measured capacities/targets -> Step11O archived targets; no raw CALCE re-fit or native-export provenance claimed'}
(OUTPUT_RUN/'CALCE_Validation_Report.json').write_text(json.dumps(report,indent=2,default=str))
(OUTPUT_RUN/'README_LIMITATIONS.txt').write_text('CALCE source workbook verification from the preserved 231 XLSX files using exact Step11A target-reconstruction functions. This is a source-to-target audit, not a full feature-engineering/model refit. Source workbooks themselves are not redistributed.\n')
with ZipFile(result_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=8) as z:
    for p in sorted(OUTPUT_RUN.iterdir()):
        if p.is_file():z.write(p,arcname=p.name)
print('\nSaved result archive:',result_zip,'bytes:',result_zip.stat().st_size)
if not report['all_pass']:
    print('FAILURES RECORDED: upload the ZIP for diagnosis; do not alter frozen inputs.')
    raise RuntimeError('One or more CALCE source comparisons failed. The result archive has been saved.')
print('PASS: eight CALCE source-workbook audits and CALCE Step11O target linkage.')
print('LIMITATION: does not verify source XLSX creation from original acquisition logs or refit historical models.')
